In [0]:
%sql
-- STEP 1 — VERIFY CDC ENABLEMENT
DESCRIBE HISTORY retail_lakehouse.silver.customers;

In [0]:
%sql
-- STEP 2 — SIMULATE CUSTOMER CHANGE
UPDATE retail_lakehouse.silver.customers
SET City = 'Bangalore'
WHERE CustomerID = 1;

In [0]:
%sql
-- STEP 3 — VIEW CDC CHANGES
SELECT
    _change_type,
    _commit_version,
    CustomerID,
    CustomerName,
    City
FROM table_changes(
    'retail_lakehouse.silver.customers',
    1
)
ORDER BY _commit_version DESC;

In [0]:
%sql
-- STEP 4 — EXPIRE OLD ACTIVE RECORD
MERGE INTO retail_lakehouse.gold.dim_customer tgt
USING (
    SELECT DISTINCT
        CustomerID
    FROM table_changes(
        'retail_lakehouse.silver.customers',
        1
    )
    WHERE _change_type = 'update_postimage'
) src
ON tgt.CustomerID = src.CustomerID
AND tgt.IsActive = TRUE

WHEN MATCHED THEN
UPDATE SET
    tgt.EndDate = CURRENT_DATE(),
    tgt.IsActive = FALSE;

In [0]:
%sql
-- STEP 5 — INSERT NEW ACTIVE VERSION
INSERT INTO retail_lakehouse.gold.dim_customer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,
    CURRENT_DATE(),
    DATE '9999-12-31',
    TRUE
FROM (
    SELECT *
    FROM table_changes(
        'retail_lakehouse.silver.customers',
        1
    )
    WHERE _change_type = 'update_postimage'
) src
LEFT JOIN retail_lakehouse.gold.dim_customer tgt
ON src.CustomerID = tgt.CustomerID
AND tgt.IsActive = TRUE
WHERE tgt.CustomerID IS NULL;

In [0]:
%sql
-- STEP 6 — VALIDATE SCD TYPE 2
SELECT
    CustomerID,
    CustomerName,
    City,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1
ORDER BY StartDate;

In [0]:
%sql
-- STEP 7 — VALIDATE ACTIVE vs INACTIVE
SELECT
    CustomerID,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

In [0]:
%sql
-- STEP 8 — VALIDATE ONLY ONE ACTIVE ROW
SELECT
    CustomerID,
    COUNT(*)
FROM retail_lakehouse.gold.dim_customer
WHERE IsActive = TRUE
GROUP BY CustomerID
HAVING COUNT(*) > 1;

In [0]:
%sql
-- STEP 9 — HISTORICAL VALIDATION
SELECT *
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

In [0]:
%sql
SELECT *
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1 AND IsActive = FALSE;